# Practical 10 - ML Monitoring with Prometheus and Grafana
Course: RTAI-242P
---

In [ ]:
import os, joblib, json, time, random
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

os.makedirs('models', exist_ok=True)
iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_tr_sc, y_tr)
acc = accuracy_score(y_te, model.predict(X_te_sc))
print(f'Accuracy: {acc:.4f}')
joblib.dump(model, 'models/iris_model.pkl')
joblib.dump(scaler, 'models/iris_scaler.pkl')
json.dump(iris.target_names.tolist(), open('models/class_names.json','w'))
print('Models saved!')

## Prometheus Metric Types

In [ ]:
from prometheus_client import Counter, Histogram, Gauge, Summary, CollectorRegistry, generate_latest

reg = CollectorRegistry()
pred_counter = Counter('predictions_total', 'Total predictions', ['status'], registry=reg)
latency_hist = Histogram('inference_latency_seconds', 'Inference time', buckets=[0.001,0.005,0.01,0.05,0.1,0.5], registry=reg)
drift_gauge = Gauge('model_drift_score', 'Drift score', registry=reg)
confidence_gauge = Gauge('model_confidence', 'Confidence', registry=reg)
print('Metric types: Counter, Histogram, Gauge created!')

In [ ]:
model = joblib.load('models/iris_model.pkl')
scaler = joblib.load('models/iris_scaler.pkl')
class_names = json.load(open('models/class_names.json'))
samples = [[5.1,3.5,1.4,0.2],[6.0,2.9,4.5,1.5],[6.3,3.3,6.0,2.5]]

for i in range(15):
    s = random.choice(samples)
    f = [[x + random.uniform(-0.2,0.2) for x in s]]
    f_sc = scaler.transform(f)
    t0 = time.time()
    pred = model.predict(f_sc)[0]
    proba = model.predict_proba(f_sc)[0]
    lat = time.time()-t0
    pred_counter.labels(status='success').inc()
    latency_hist.observe(lat)
    drift_gauge.set(random.uniform(0.1,0.5))
    confidence_gauge.set(proba.max())
    print(f'[{i+1:2d}] {class_names[pred]:<12} conf={proba.max():.3f} lat={lat*1000:.3f}ms')

In [ ]:
raw = generate_latest(reg).decode('utf-8')
for line in raw.split('\n'):
    if line.strip() and not line.startswith('#'):
        print(line)

In [ ]:
import matplotlib.pyplot as plt

time_points = list(range(50))
drift_scores = []
current = 0.1
for t in time_points:
    current = max(0, min(1, current + random.uniform(-0.01, 0.03)))
    drift_scores.append(current)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(time_points, drift_scores, 'b-', linewidth=2)
axes[0].axhline(y=0.3, color='orange', linestyle='--', label='Warning 0.3')
axes[0].axhline(y=0.6, color='red', linestyle='--', label='Critical 0.6')
axes[0].fill_between(time_points, 0.6, drift_scores, where=[d>0.6 for d in drift_scores], color='red', alpha=0.3)
axes[0].set_title('Model Drift Score (Gauge)'); axes[0].legend(); axes[0].set_ylim(0,1)

latencies = [random.expovariate(100)*1000 for _ in range(500)]
axes[1].hist(latencies, bins=40, color='steelblue', alpha=0.7)
axes[1].axvline(x=50, color='orange', linestyle='--', label='50ms')
axes[1].axvline(x=100, color='red', linestyle='--', label='100ms')
axes[1].set_title('Inference Latency Distribution (Histogram)'); axes[1].legend()
plt.tight_layout(); plt.show()
print('Charts displayed!')

## PromQL Query Examples
```promql
# Total predictions
sum(ml_predictions_total)

# Request rate
rate(ml_predictions_total[1m])

# Error rate %
sum(rate(ml_prediction_errors_total[5m])) / sum(rate(ml_predictions_total[5m])) * 100

# Average latency ms
rate(ml_inference_duration_seconds_sum[5m]) / rate(ml_inference_duration_seconds_count[5m]) * 1000

# P95 latency
histogram_quantile(0.95, rate(ml_inference_duration_seconds_bucket[5m])) * 1000

# Drift score
ml_model_drift_score
```


## Summary

| Component | Port | Role |
|---|---|---|
| FastAPI | 8000 | ML model + /metrics |
| Prometheus | 9090 | Scrape + store metrics |
| Grafana | 3000 | Visualize + alert |
| Streamlit | 8501 | Local dashboard |

RTAI-242P Practical 10 Complete!